# Mini Project B - Titanic Survival Prediction

## Objective

The objective of this project is to build a Machine Learning model capable of predicting whether a passenger survived the Titanic disaster.

This project builds upon the Exploratory Data Analysis (EDA) completed in Mini Project A. The insights obtained during EDA are used to guide feature selection, preprocessing, and model development.

### Workflow

- Load Dataset
- Perform Feature Engineering
- Apply Data Preprocessing
- Train Multiple Machine Learning Models
- Evaluate Model Performance
- Save the Best Model using Joblib

In [13]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

import joblib
import time

In [14]:
df = pd.read_csv("../Titanic-Dataset.csv")

print("Dataset Loaded Successfully")
print(f"Shape : {df.shape}")

df.head()

Dataset Loaded Successfully
Shape : (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [15]:
print(df.info())

display(df.describe())

display(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
None


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [16]:
# Family Size
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# Is Alone
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

# Extract Title
df["Title"] = df["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)

# Simplify Titles
df["Title"] = df["Title"].replace(
    ['Lady','Countess','Capt','Col','Don','Dr','Major',
     'Rev','Sir','Jonkheer','Dona'],
    'Rare'
)

df["Title"] = df["Title"].replace({
    'Mlle':'Miss',
    'Ms':'Miss',
    'Mme':'Mrs'
})

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize,IsAlone,Title
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,2,0,Mr
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,2,0,Mrs
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1,1,Miss
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,2,0,Mrs
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,1,1,Mr


In [17]:
# Drop unnecessary columns

df = df.drop(columns=[
    "PassengerId",
    "Name",
    "Ticket",
    "Cabin"
])

X = df.drop("Survived", axis=1)
y = df["Survived"]

print(X.head())

   Pclass     Sex   Age  SibSp  Parch     Fare Embarked  FamilySize  IsAlone  \
0       3    male  22.0      1      0   7.2500        S           2        0   
1       1  female  38.0      1      0  71.2833        C           2        0   
2       3  female  26.0      0      0   7.9250        S           1        1   
3       1  female  35.0      1      0  53.1000        S           2        0   
4       3    male  35.0      0      0   8.0500        S           1        1   

  Title  
0    Mr  
1   Mrs  
2  Miss  
3   Mrs  
4    Mr  


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Samples :", X_train.shape)
print("Testing Samples  :", X_test.shape)

Training Samples : (712, 10)
Testing Samples  : (179, 10)


In [19]:
numeric_features = [
    "Age",
    "Fare",
    "SibSp",
    "Parch",
    "FamilySize",
    "IsAlone"
]

categorical_features = [
    "Sex",
    "Embarked",
    "Title",
    "Pclass"
]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [20]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=200,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    ),

    "AdaBoost": AdaBoostClassifier(
        random_state=42
    ),

    "K-Nearest Neighbors": KNeighborsClassifier(),

    "Support Vector Machine": SVC(
        probability=True,
        random_state=42
    ),

    "Gaussian Naive Bayes": GaussianNB()
}

In [21]:
results = []

trained_models = {}

for name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", model)
        ]
    )

    # Measure Training Time
    start = time.time()
    pipeline.fit(X_train, y_train)
    training_time = time.time() - start

    predictions = pipeline.predict(X_test)
    # Compute ROC-AUC probabilities
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    roc_auc = roc_auc_score(y_test, probabilities)

    # Add Cross Validation
    cv_score = cross_val_score(
        pipeline,
        X,
        y,
        cv=5,
        scoring="accuracy"
    )
    cv_mean = cv_score.mean()

    trained_models[name] = pipeline

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc,
        "CV Accuracy": cv_mean,
        "Training Time (s)": training_time
    })

In [22]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

display(results_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,CV Accuracy,Training Time (s)
0,Logistic Regression,0.854749,0.841270,0.768116,0.803030,0.878656,0.822673,0.032250
7,Support Vector Machine,0.843575,0.825397,0.753623,0.787879,0.845059,0.833877,0.062177
6,K-Nearest Neighbors,0.821229,0.793651,0.724638,0.757576,0.853030,0.815988,0.012044
8,Gaussian Naive Bayes,0.821229,0.760563,0.782609,0.771429,0.858235,0.804714,0.013532
4,Gradient Boosting,0.815642,0.810345,0.681159,0.740157,0.850527,0.829408,0.163782
2,Random Forest,0.810056,0.769231,0.724638,0.746269,0.823386,0.795744,0.318222
3,Extra Trees,0.810056,0.769231,0.724638,0.746269,0.809816,0.781156,0.271564
5,AdaBoost,0.810056,0.739726,0.782609,0.760563,0.854743,0.813703,0.121394
1,Decision Tree,0.804469,0.757576,0.724638,0.740741,0.778458,0.765426,0.012028


In [23]:
best_model_name = results_df.iloc[0]["Model"]

best_model = trained_models[best_model_name]

joblib.dump(best_model, "model.pkl")

print(f"Best Model : {best_model_name}")

print("Saved Successfully as model.pkl")

Best Model : Logistic Regression
Saved Successfully as model.pkl


In [24]:
loaded_model = joblib.load("model.pkl")

sample_prediction = loaded_model.predict(X_test.iloc[:5])

print("Predictions")

print(sample_prediction)

print("Actual Values")

print(y_test.iloc[:5].values)

Predictions
[0 0 0 0 1]
Actual Values
[0 0 1 0 1]


In [25]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=["Accuracy", "F1 Score"],
    ascending=False
)

results_df = results_df.reset_index(drop=True)

display(results_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,CV Accuracy,Training Time (s)
0,Logistic Regression,0.854749,0.841270,0.768116,0.803030,0.878656,0.822673,0.032250
1,Support Vector Machine,0.843575,0.825397,0.753623,0.787879,0.845059,0.833877,0.062177
2,Gaussian Naive Bayes,0.821229,0.760563,0.782609,0.771429,0.858235,0.804714,0.013532
3,K-Nearest Neighbors,0.821229,0.793651,0.724638,0.757576,0.853030,0.815988,0.012044
4,Gradient Boosting,0.815642,0.810345,0.681159,0.740157,0.850527,0.829408,0.163782
5,AdaBoost,0.810056,0.739726,0.782609,0.760563,0.854743,0.813703,0.121394
6,Random Forest,0.810056,0.769231,0.724638,0.746269,0.823386,0.795744,0.318222
7,Extra Trees,0.810056,0.769231,0.724638,0.746269,0.809816,0.781156,0.271564
8,Decision Tree,0.804469,0.757576,0.724638,0.740741,0.778458,0.765426,0.012028


## Conclusion

Three classification models were trained and evaluated:

- Logistic Regression
- Decision Tree
- Random Forest

The best-performing model was selected automatically based on accuracy and saved as `model.pkl`.

This saved pipeline includes:

- Missing value imputation
- Feature scaling
- One-hot encoding
- Machine Learning classifier

The saved model will be used in the next phase to build a FastAPI application and expose a prediction API.